In [1]:
!poetry install -q

In [18]:
"""
환경 설정 및 의존성 주입
- 목적: 프로젝트 경로 인식, 환경 변수 로드, 그리고 로컬 테스트를 위한 Docker DNS 우회
"""
import os
import sys
import pandas as pd
from dotenv import load_dotenv

# 1. 프로젝트 경로 설정 및 환경 변수 명시적 로드
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

env_path = os.path.join(project_root, '.env')
load_dotenv(dotenv_path=env_path)

# [핵심 수정 사항] 
# Docker 네트워크 외부(Host OS)에서 실행되는 Jupyter를 위한 DNS 해석 우회 처리
local_s3_endpoint = os.environ.get("LOCAL_S3_ENDPOINT", "")
if "localstack" in local_s3_endpoint:
    os.environ["LOCAL_S3_ENDPOINT"] = local_s3_endpoint.replace("localstack", "localhost")

# 2. 모듈 임포트
from src.common.config import ConfigManager
from src.reader.reader_service import ReaderService
from src.transformer.transformer_service import TransformerService

print("✅ 환경 설정 및 모듈 임포트 완료")

✅ 환경 설정 및 모듈 임포트 완료


In [ ]:
# DataFrame 출력 생략 방지 옵션 설정
pd.set_option('display.max_columns', None)        # 숨김 없이 모든 컬럼 출력
pd.set_option('display.max_colwidth', None)       # 컬럼 안의 긴 텍스트(Dict/List) 전체 출력
pd.set_option('display.expand_frame_repr', False) # 가로 너비 초과 시 줄바꿈 방지
pd.set_option('display.max_rows', 50)             # 필요시 최대 출력 행 수 조정

In [19]:
"""
5종 API 파티션 데이터 스트리밍 추출
- 목적: KIS, FRED, ECOS, UPBIT의 대표 job_id를 순회하며 파티션 단위 스트리밍을 통해 원본 DataFrame을 획득합니다.
"""

# 1. ReaderService 초기화
reader = ReaderService(target_reader="s3")

# 2. 테스트할 프로바이더 및 job_id 리스트 정의
target_jobs = [
    ("kis", "kis_kospi_finance_daily"),
    ("kis", "kis_nasdaq_daily"),
    ("fred", "fred_us_treasury_10y_daily"),
    ("ecos", "ecos_ktb_10y_daily"),
    ("upbit", "upbit_krw_btc_daily")
]

# 공통 파티션 날짜 (테스트 기준일)
base_date_path = "year=2026/month=05/day=06"

# 3. 데이터 추출 및 확인
raw_dataframes = {}

for provider, job_id in target_jobs:
    # 파티션 기반 동적 S3 Key(Prefix) 생성
    s3_key = f"raw/provider={provider}/job={job_id}/{base_date_path}"
    
    records = []
    try:
        # Paginator를 통한 스트림 추출
        for batch in reader.read_stream(source_path=s3_key):
            records.extend(batch)
            
        df = pd.DataFrame(records)
        raw_dataframes[job_id] = df  # 다음 셀(변환 테스트)에서 사용하기 위해 딕셔너리에 저장
        
        print(f"\n[{job_id}] 원본 데이터 추출 성공 - 총 레코드 수: {len(df)}")
        #display(df.head(5))
        print(df.loc[:5])

    except Exception as e:
        print(f"❌ 추출 실패: {e}")

[2026-05-07 06:31:24]  INFO   | ReaderService        | [8a969c0c] 데이터 스트림 추출 요청 위임 - Target: S3, Path: raw/provider=kis/job=kis_kospi_finance_daily/year=2026/month=05/day=06, Batch: 10000
[2026-05-07 06:31:24]  INFO   | ReaderService        | [8a969c0c] [S3] 리더 인스턴스 지연 초기화 진입
[2026-05-07 06:31:24]  INFO   | S3ZstdStreamingReader | [8a969c0c] [S3_BRONZE_READER] LocalStack S3 Endpoint로 클라이언트 초기화 (http://localhost:4566)
[2026-05-07 06:31:24]  INFO   | S3ZstdStreamingReader | [8a969c0c] [S3_BRONZE_READER] S3 스트리밍 읽기 시작 - Bucket: data-pipeline-bronze, Key: raw/provider=kis/job=kis_kospi_finance_daily/year=2026/month=05/day=06
[2026-05-07 06:31:24]  INFO   | S3ZstdStreamingReader | [8a969c0c] [S3_BRONZE_READER] S3 파티션 스트리밍 완료 - 처리된 파일: 2개, 총 레코드: 2건

[kis_kospi_finance_daily] 원본 데이터 추출 성공 - 총 레코드 수: 2
                                                                                                                                                                                                 

In [20]:
"""
[Cell 3] 5종 API 통합 변환(Transformer) 테스트
- 목적: Cell 2에서 추출한 원본 데이터를 스트리밍으로 전달하여 Silver 데이터로 변환합니다.
- 특징: S3 재호출 없이 메모리 내의 DataFrame을 제너레이터로 래핑(iter)하여 파이프라인을 고속 모사합니다.
"""
from src.transformer.transformer_service import TransformerService
import pandas as pd
from IPython.display import display

# 1. Transformer 서비스 초기화
transformer_service = TransformerService()
ENFORCE_SCHEMA = True  # Production 모드 (Silver 스키마 강제 적용)

for job_id, raw_df in raw_dataframes.items():
    print(f"\n" + "━" * 80)
    print(f"🚀 [변환 파이프라인] {job_id}")
    print("━" * 80)
    
    try:
        # 2. DataFrame을 단일 청크 스트림(Iterator)으로 래핑하여 메모리 낭비 없이 제너레이터 주입
        raw_stream = iter([raw_df])
        
        # 3. 스트리밍 변환 실행
        transformed_stream = transformer_service.transform_stream(
            job_id=job_id, 
            data_stream=raw_stream,
            enforce_schema=ENFORCE_SCHEMA
        )
        
        # 4. 결과 병합 (리스트 컴프리헨션 대체로 코드 간결화)
        transformed_chunks = list(transformed_stream)
        
        if transformed_chunks:
            silver_df = pd.concat(transformed_chunks, ignore_index=True)
            print(f"✅ 변환 성공 | 크기: {len(silver_df)} 행 × {len(silver_df.columns)} 열")
            print(f"📌 데이터 타입 스키마:\n{silver_df.dtypes.to_string()}\n")
            display(silver_df.head(2))
        else:
            print("⚠️ 변환된 데이터가 없습니다 (빈 데이터프레임).")
            
    except Exception as e:
        # 아직 변환기가 구현되지 않은 API(FRED, ECOS 등)는 여기서 명확히 에러를 뱉어냅니다.
        print(f"❌ 변환 실패 (Fail-Fast 정상 작동): {e}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 [변환 파이프라인] kis_kospi_finance_daily
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[2026-05-07 06:31:24]  INFO   | TransformerService   | [8a969c0c] [kis_kospi_finance_daily] 변환기 지연 초기화 진입 (Policy: kis_domestic_schema)
[2026-05-07 06:31:24]  INFO   | TransformerService   | [8a969c0c] [kis_kospi_finance_daily] 스트리밍 데이터 변환 파이프라인 가동 (enforce_schema=True)
[2026-05-07 06:31:24]  INFO   | TransformerService   | [8a969c0c] [kis_kospi_finance_daily] 스트리밍 변환 완료 (총 1개 청크 처리됨)
✅ 변환 성공 | 크기: 2 행 × 13 열
📌 데이터 타입 스키마:
price_change_sign     string
close                float32
open                 float32
high                 float32
low                  float32
prev_price           float32
price_change_rate    float32
volume               float32
prev_volume          float32
trading_value        float32
futures_prev_open    float32
futures_prev_high    float32
futures_prev_low     floa

,price_change_sign,close,open,high,low,prev_price,price_change_rate,volume,prev_volume,trading_value,futures_prev_open,futures_prev_high,futures_prev_low
0,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 [변환 파이프라인] kis_nasdaq_daily
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[2026-05-07 06:31:24]  INFO   | TransformerService   | [8a969c0c] [kis_nasdaq_daily] 변환기 지연 초기화 진입 (Policy: kis_overseas_schema)
[2026-05-07 06:31:24]  INFO   | TransformerService   | [8a969c0c] [kis_nasdaq_daily] 스트리밍 데이터 변환 파이프라인 가동 (enforce_schema=True)
[2026-05-07 06:31:24]  ERROR  | KISTransformer       | [8a969c0c] [KISTransformer] 변환 로직 수행 중 예기치 않은 오류 발생 | Error: unhashable type: 'list' (상세 내용은 JSON 파일 참조)
[2026-05-07 06:31:24]  ERROR  | AbstractTransformer  | [8a969c0c] [AbstractTransformer.transform] FAILED | Time: 0.0007s | Type: TransformerError | Retry: False (상세 내용은 JSON 파일 참조)
❌ 변환 실패 (Fail-Fast 정상 작동): [TransformerError] 스트리밍 변환 제너레이터 실행 중 오류 발생: [TransformerError] [KISTransformer] 변환 로직 수행 중 예기치 않은 네이티브 오류 발생 (Caused by: TypeError) (Caused by: TransformerError)

━━━━━━━━━━━━━━━━━━

""
0
1



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🚀 [변환 파이프라인] ecos_ktb_10y_daily
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[2026-05-07 06:31:24]  INFO   | TransformerService   | [8a969c0c] [ecos_ktb_10y_daily] 변환기 지연 초기화 진입 (Policy: ecos_base_schema)
[2026-05-07 06:31:24]  INFO   | TransformerService   | [8a969c0c] [ecos_ktb_10y_daily] 스트리밍 데이터 변환 파이프라인 가동 (enforce_schema=True)
[2026-05-07 06:31:24]  ERROR  | ECOSTransformer      | [8a969c0c] [ECOSTransformer] 변환 로직 수행 중 예기치 않은 오류 발생 | Error: All items in data must be of type dict, found flo... (상세 내용은 JSON 파일 참조)
[2026-05-07 06:31:24]  ERROR  | AbstractTransformer  | [8a969c0c] [AbstractTransformer.transform] FAILED | Time: 0.0016s | Type: TransformerError | Retry: False (상세 내용은 JSON 파일 참조)
❌ 변환 실패 (Fail-Fast 정상 작동): [TransformerError] 스트리밍 변환 제너레이터 실행 중 오류 발생: [TransformerError] [ECOSTransformer] 변환 로직 수행 중 예기치 않은 네이티브 오류 발생 (Caused by: TypeError) (Caused by: Tra

""
0
1
